# cadence 学習フェーズ【Colab・要 GPU・CV / CSJ 両対応】

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/Style-Bert-VITS2/blob/layer-b-cadence-seq/colab/cadence_train_colab.ipynb)

**学習バンドル（tar）1 本を入力に、bert_gen → style_gen → 学習まで通す。**
コーパスは §1.2 の `CORPUS` で切り替える（既定 `cv_r1`）。バンドルの仕様は
[`docs/TRAIN_BUNDLE_SPEC.md`](../docs/TRAIN_BUNDLE_SPEC.md)。

| CORPUS | 取得 | ライセンス |
|---|---|---|
| `cv_r1` | Zenodo から**自動 DL**（DOI 10.5281/zenodo.21119791） | データ CC0 / 重みは AGPL-3.0 で公開可 |
| `csj_r1` | **自動 DL しない**。各自が CSJ 現物から作った tar を `DRIVE_BASE/corpus_in/` に置く | **CSJ は二次配布禁止**。tar も学習済み重みも公開不可（重みの公開は NINJAL 事前確認が必要） |

**前提**: `cadence_train_setup_colab.ipynb` を先に 1 回実行（fork clone + 底モデル取得）。

**規約**: 話者数・発話数・行数は**すべて config.json と esd から読む**。
ノートに数を焼き込まない（焼き込むと別コーパスで必ず破れる）。


In [ ]:
# ===== §1 マウント・前提チェック【毎回・ランタイム再起動後も最初に実行】=====
from google.colab import drive
from pathlib import Path
import os, subprocess

DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")   # setup と同じ値（clone の置き場所）

def _drive_alive():
    """マウントの生死確認（stale mount だと listdir が OSError: Transport endpoint is not connected）"""
    try:
        next(iter(os.listdir("/content/drive/MyDrive")), None)
        return True
    except OSError:
        return False

drive.mount("/content/drive")
if not _drive_alive():
    print("★Drive マウントが切れている（Transport endpoint is not connected）→ 強制再マウント")
    subprocess.run(["fusermount", "-u", "/content/drive"], capture_output=True)
    drive.mount("/content/drive", force_remount=True)
    assert _drive_alive(), "★再マウント失敗 → ランタイム再起動してやり直す"
assert DRIVE_BASE.exists(), f"★Drive に fork clone が無い: {DRIVE_BASE}（先に cadence_train_setup_colab.ipynb を実行）"

# --- fork の更新を Drive clone に自動反映（ベストエフォート。失敗しても続行）---
# バッジで開くノートは常に GitHub の最新だが、コードは Drive clone のスナップショット。
# ここで揃えないと「新しいノート × 古いコード」の不整合が起きる。意図的に版を固定したい場合は False。
AUTO_PULL = True
if AUTO_PULL:
    try:
        _p = subprocess.run(["git", "-C", str(DRIVE_BASE), "pull", "--ff-only"],
                            capture_output=True, text=True, timeout=180)
        if _p.returncode == 0:
            print("git pull:", (_p.stdout.strip().splitlines() or ["?"])[-1])
        else:
            print("★git pull 失敗（そのまま続行。clone の手元変更やネットワークを確認）:",
                  (_p.stderr or "").strip()[-200:])
    except Exception as e:
        print("★git pull 例外（そのまま続行）:", e)
_h = subprocess.run(["git", "-C", str(DRIVE_BASE), "rev-parse", "--short", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("clone commit:", _h)

os.chdir(DRIVE_BASE); print("cwd:", Path.cwd())   # 依存インストールは requirements.txt をここから読む（後段でローカルへ移る）

br = subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip()
assert br == "layer-b-cadence-seq", f"★branch が違う: {br} → git checkout layer-b-cadence-seq"
print("branch:", br)

g = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
assert g.returncode == 0 and "GPU" in g.stdout, "★GPU ランタイムでない → [ランタイム]→[ランタイムのタイプを変更]→GPU"
print(g.stdout.strip())


In [ ]:
# ===== §1.2 コーパス選択【毎回・§1 の直後に実行】=====
# ★このノートで CV / CSJ を切り替える唯一の場所。以降のセルは CORPUS しか見ない。
CORPUS = "cv_r1"          # "cv_r1" | "csj_r1"

# 取得元の定義。新コーパスはここに 1 エントリ足すだけで通る（ノート本体は無改修）。
#   fetch="zenodo": 公開 URL から自動 DL      → 配布可能なデータ専用
#   fetch="local" : DRIVE_BASE/corpus_in/ に各自が置いた tar を使う（URL を持たない = 配布しない）
CORPORA = {
    "cv_r1": {
        "label": "Common Voice ja (CC0)",
        "fetch": "zenodo",
        "files": [   # (ファイル名, URL, 完全性の下限バイト数)
            ("cadence_cv_r1_meta_v20260702.tgz",
             "https://zenodo.org/records/21119791/files/cadence_cv_r1_meta_v20260702.tgz?download=1",
             30_000_000),
            ("cadence_cv_r1_wavs_v20260702.tar",
             "https://zenodo.org/records/21119791/files/cadence_cv_r1_wavs_v20260702.tar?download=1",
             5_000_000_000),
        ],
        "smoke": ("cadence_cv_r1_smoke_v1.tgz",
                  "https://github.com/slp-hu/Style-Bert-VITS2/releases/download/"
                  "cv_r1-smoke-v1/cadence_cv_r1_smoke_v1.tgz",
                  100_000_000),
        "license": "データ CC0（Zenodo DOI 10.5281/zenodo.21119791）/ 学習済み重みは AGPL-3.0 で公開可",
    },
    # --- CSJ 系（各自でライセンス取得・URL を持たない = コードから配布が起き得ない）------
    # tar は dataset_tools/pack_bundle.py が作る。手で作らないこと（docs/TRAIN_BUNDLE_SPEC.md）。
    "csj_core75": {
        "label": "CSJ コア 75 話者（配管検証用。R1 ではない）",
        "fetch": "local",
        "files": [("csj_core75__REPLACE_meta.tgz", None, 1_000_000),
                  ("csj_core75__REPLACE_wavs.tar", None, 1_000_000_000)],
        "smoke": None,     # 公開 Releases に置けない（CSJ 由来物のため）
        "license": ("★CSJ は二次配布禁止。この tar を再配布しないこと。"
                    "学習済み重み・合成音声の公開も NINJAL の事前確認が必要"),
    },
    "csj_r1": {
        "label": "CSJ 375 話者（R1 = e10_s83230 の学習データ）",
        "fetch": "local",
        "files": [("csj_r1__cdae12ce_meta.tgz", None, 100_000_000),
                  ("csj_r1__cdae12ce_wavs.tar", None, 50_000_000_000)],
        "smoke": None,
        "license": ("★CSJ は二次配布禁止。この tar を再配布しないこと。"
                    "学習済み重み・合成音声の公開も NINJAL の事前確認が必要"),
    },
}
# tar の置き場。既定は DRIVE_BASE/corpus_in/ だが、既にどこかにあるなら動かさずここを変える。
LOCAL_IN = DRIVE_BASE / "corpus_in"        # 例: Path("/content/drive/MyDrive/SBV2")

assert CORPUS in CORPORA, f"★未知の CORPUS: {CORPUS}（{list(CORPORA)} のいずれか）"
C = CORPORA[CORPUS]
print(f"CORPUS = {CORPUS}  ({C['label']})")
print("ライセンス:", C["license"])
if C["fetch"] == "local":
    print(f"\n★入力 tar は自動取得しない。次の場所に置いてから続行:")
    for name, _, _ in C["files"]:
        print(f"   {LOCAL_IN}/{name}")
    print("   （作り方: dataset_tools/ + docs/TRAIN_BUNDLE_SPEC.md）")


## §1.5 スモーク学習（動作確認）設定 — 無料 Colab 用

`SMOKE = True` にすると、**そのコーパスのスモークバンドル**（§1.2 の `CORPORA[...]["smoke"]`）で
配管だけを通す。`smoke` が `None` のコーパス（= 公開 Releases に置けない `csj_r1` 等）では
`SMOKE=True` は assert で止まる。

- checkpoint はスモーク専用のローカルツリー `Data/{CORPUS}_smoke/models/`（VM 揮発）に保存され、
  **Drive の本番 `models/` には書かない**（本番 checkpoint からの誤再開も起きない）。
  合成用モデルも `model_assets/{CORPUS}_smoke/` に分離される。
- バンドル内の config は全量と同一 spk2id を維持し、底モデルもそのまま使うため、
  モデル形状・warm-start の検証としては本番と等価。
- compute capability < 8.0 の GPU（T4 = 7.5 等）では bf16/fp16 を自動で無効化する（fp32）。
- **学習品質の評価には使えない**（step 数が2桁足りない）。目的は配管の検証のみ。
- cv_r1 のバンドルは `colab/make_smoke_bundle.py` で全量 `Data/cv_r1` から決定的に再生成できる
  （GitHub Releases: tag `cv_r1-smoke-v1` / アセット名固定。§1.2 の URL がこれを指す）。


In [ ]:
# ===== §1.5 スモーク学習（動作確認）設定 =====
SMOKE = False            # ← 動作確認するときだけ True（本番学習は False のまま）
SMOKE_EPOCHS        = 2
SMOKE_BATCH         = 4    # T4 (16GB) の fp32 実測で 8 は OOM（WavLM 判別器が重い）。それでも溢れたら 2 に
SMOKE_EVAL_INTERVAL = 100  # 保存間隔（既定 1000 のままだと保存前に学習が終わる）
assert not (SMOKE and C["smoke"] is None), (
    f"★{CORPUS} にはスモークバンドルが無い（公開 Releases に置けないコーパス）。"
    " SMOKE=False で本番バンドルを使うか、少数話者の tar を別途作って CORPORA に足す")
print("SMOKE =", SMOKE, f"(CORPUS={CORPUS})")


In [ ]:
# ===== §2 GPU 判定と依存インストール【毎セッション実行・再起動後も再実行（冪等）】=====
# GPU の compute capability で経路を自動分岐する:
#   ・sm_90 以下（T4/L4/A100 等）: requirements の pin どおり torch 2.3.1(cu121)。再起動不要。
#   ・sm_100 以上（Blackwell 系: RTX PRO 6000 = sm_120 等）: torch 2.3.1 は sm_90 までで非対応
#     （GPU forward で落ちる）→ torch 2.11.0+cu128 へ入替（07a 方式）。★入替後はランタイム再起動が必須。
# 再起動後にこのセルを再実行すると、入替をスキップして検証だけ行う。
import subprocess, sys, re, os
from pathlib import Path

def _run(cmd, stream=False):
    print("$", " ".join(cmd))
    if stream:   # 進捗をそのまま流す（★torch の数GB DL は -q だと無言=フリーズと誤認するため）
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout: print(line, end="")
        p.wait(); assert p.returncode == 0, f"失敗: {cmd}"
    else:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or "")[-1200:] or "(quiet)")
        if r.returncode != 0:
            print(r.stderr[-4000:]); raise SystemExit(f"失敗: {cmd}")

# --- GPU capability（torch を import せず nvidia-smi で判定するのが肝）---
cap = subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
assert cap and cap[0], "★GPU が見えない（GPU ランタイムか確認）"
CAP = float(cap[0]); IS_BLACKWELL = CAP >= 10.0
print(f"compute capability: {cap[0]} (sm_{int(CAP*10)}) → 経路: "
      + ("Blackwell（torch 2.11+cu128 入替）" if IS_BLACKWELL else "標準（torch 2.3.1 のまま）"))

# --- 現在の torch を subprocess で確認（in-kernel import しない = 再起動不要性を保つ）---
q = subprocess.run([sys.executable,"-c","import torch;print(torch.__version__)"], capture_output=True, text=True)
TORCH_NOW = q.stdout.strip()
print("現在の torch:", TORCH_NOW or "(未導入)")
ALREADY_SWAPPED = IS_BLACKWELL and TORCH_NOW.startswith("2.11.")

if ALREADY_SWAPPED:
    print("→ 再起動後の再実行と判断: torch 入替済みのため install をスキップ")
else:
    # (1) requirements から faster-whisper を除外して install（07a 方式）。
    #     faster-whisper==0.10.1 が av==10.* をソースビルドしようとし Py3.12 で失敗するため。
    #     文字起こし用途で cv_r1（bert_gen/style_gen/train/eval）には不要。
    req = Path("requirements.txt"); req_train = Path("requirements_no_whisper.txt")
    _drop = re.compile(r"^(faster-whisper|av)==")   # ビルドで詰まるパッケージが増えたらここに追加（例: stable_ts）
    kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
    req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
    _run([sys.executable,"-m","pip","install","-q","-r",str(req_train)])

    # Colab 既載の torchvision 等は torch 2.11 用ビルドのまま残り、torch 2.3.1 環境では壊れている
    # （torch.library.register_fake は 2.4+）。SBV2 は不使用で、残っていると umap 等が import して
    # 落ちるため standard 経路でも外す（Blackwell 経路と同じ扱い）。
    _run([sys.executable,"-m","pip","uninstall","-y","-q",
          "torchvision","torchcodec","torchao","torchtune","torchdata"])

    # (2) Blackwell のみ: torch 2.11.0+cu128 へ入替（実績のある手順をそのまま踏む）
    if IS_BLACKWELL:
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torch","torchaudio","torchvision","torchcodec","torchao","torchtune","torchdata"])
        _run([sys.executable,"-m","pip","install","torch==2.11.0","torchaudio==2.11.0",
              "--index-url","https://download.pytorch.org/whl/cu128"], stream=True)   # ★-q 禁止
        _run([sys.executable,"-m","pip","install","-q","soundfile"])
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torchcodec","torchvision","torchao","torchtune","torchdata"])

    # (3) HF スタック固定（transformers 未ピン → Colab 既定の transformers 5.x が torch>=2.4 を要求して
    #     torch を無効化し、bert_gen が "AutoModelForMaskedLM requires PyTorch" で落ちる問題の恒久対策）
    _run([sys.executable,"-m","pip","install","-q",
          "transformers==4.41.2","huggingface_hub==0.23.5","tokenizers<0.20",
          "pytorch-lightning==2.2.5","torchmetrics<1.5","pyannote.audio==3.1.1",
          "scipy==1.13.1","numpy==1.26.4"])

# --- numpy 健全性チェック（混在インストールの検出と自動修復）---
# pip のダウングレードで旧 2.x の compiled .so（numpy/random/mtrand 等）が残ると、
# コアは import できるのにサブパッケージ初期化で "numpy.dtype size changed" になる。
_h = subprocess.run([sys.executable,"-c","import numpy.random, numpy; print(numpy.__version__)"],
                    capture_output=True, text=True)
if _h.returncode != 0:
    print("★numpy が混在状態（サブパッケージ初期化に失敗）→ 1.26.4 を強制再インストールで修復")
    print("  症状:", (_h.stderr or "").strip().splitlines()[-1] if _h.stderr else "?")
    _run([sys.executable,"-m","pip","install","-q","--force-reinstall","--no-deps",
          "--no-cache-dir","numpy==1.26.4"])
    _h = subprocess.run([sys.executable,"-c","import numpy.random, numpy; print(numpy.__version__)"],
                        capture_output=True, text=True)
    assert _h.returncode == 0, "★numpy を修復できない:\n" + (_h.stderr or "")[-600:]
print("numpy 健全性 OK:", _h.stdout.strip())

# --- 検証（別プロセス。transformers から torch が見えているかまで確認）---
v = subprocess.run([sys.executable,"-c",
    "import torch;from transformers.utils import is_torch_available;"
    "print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),"
    "'| transformers_sees_torch',is_torch_available())"], capture_output=True, text=True)
print(v.stdout.strip() or v.stderr[-800:])
assert "transformers_sees_torch True" in v.stdout, "★transformers が torch を認識していない → このセルをやり直す"
if (not IS_BLACKWELL) or ALREADY_SWAPPED:
    assert "cuda True" in v.stdout, "★CUDA が使えない（GPU ランタイム / torch ビルドを確認）"

if IS_BLACKWELL and not ALREADY_SWAPPED:
    print()
    print("=" * 70)
    print("★torch を入れ替えた → ここで【ランタイム再起動】が必須★")
    print("  [ランタイム] → [セッションを再起動] のあと、§1 → §2 → §3 の順に再実行してから先へ進む。")
    print("  （再起動で cwd も torch 状態もリセットされる。最初のセルを飛ばさないこと）")
    print("=" * 70)


In [ ]:
# ===== §3 torchaudio / torch.load shim【Blackwell 経路のみ実体化・毎セッション実行】=====
# torch 2.11 系では torchaudio.set_audio_backend が削除され、torchaudio.load も torchcodec 経由で壊れる。
# pyannote.audio が import 時に set_audio_backend を呼ぶため、shim なしでは style_gen / 合成が落ちる。
# `!python` で走る bert_gen / style_gen / train は【別プロセス】なので、カーネル内 monkeypatch では効かない
# → sitecustomize.py + PYTHONPATH で全 Python プロセスに注入する（ここが肝）。
import os, subprocess, sys
from pathlib import Path

if not IS_BLACKWELL:
    print("標準経路（torch 2.3.1）: shim 不要 → スキップ")
else:
    compat = Path("/content/_compat"); compat.mkdir(exist_ok=True)
    shim = compat / "sitecustomize.py"
    SHIM_SRC = '# sitecustomize: torch 2.11 環境の互換 shim（cv_r1 公開ノート用）\n# 1) torch.load の weights_only 既定を False に戻す（旧 ckpt / torch.hub モデルの読込互換）\n# 2) torchaudio.set_audio_backend / get_audio_backend を復活（pyannote.audio の import 時呼び出し対策）\n# 3) torchaudio.load / info を soundfile 実装に置換（torchcodec 経由の破綻を回避）\ntry:\n    import torch\n    _orig_torch_load = torch.load\n    def _patched_torch_load(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return _orig_torch_load(*args, **kwargs)\n    torch.load = _patched_torch_load\nexcept Exception:\n    pass\n\ntry:\n    import torchaudio\n\n    def _noop_set_backend(*args, **kwargs):\n        return None\n    def _get_backend(*args, **kwargs):\n        return "soundfile"\n    torchaudio.set_audio_backend = _noop_set_backend\n    torchaudio.get_audio_backend = _get_backend\n\n    def _sf_load(filepath, frame_offset=0, num_frames=-1, normalize=True,\n                 channels_first=True, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        import torch as _torch\n        frames = int(num_frames) if int(num_frames) > 0 else -1\n        data, sr = sf.read(str(filepath), start=int(frame_offset), frames=frames,\n                           dtype="float32", always_2d=True)\n        wav = _torch.from_numpy(data.T if channels_first else data)\n        return wav, sr\n    torchaudio.load = _sf_load\n\n    def _sf_info(filepath, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        info = sf.info(str(filepath))\n        class _AudioMetaData:\n            pass\n        meta = _AudioMetaData()\n        meta.sample_rate = info.samplerate\n        meta.num_frames = info.frames\n        meta.num_channels = info.channels\n        meta.bits_per_sample = 16\n        meta.encoding = "PCM_S"\n        return meta\n    torchaudio.info = _sf_info\nexcept Exception:\n    pass\n'
    shim.write_text(SHIM_SRC, encoding="utf-8")

    # 書き出し事故（末尾のエスケープ崩れ等 → SyntaxError）を必ず py_compile で検証する
    r = subprocess.run([sys.executable,"-m","py_compile",str(shim)], capture_output=True, text=True)
    assert r.returncode == 0, "★shim が SyntaxError: " + r.stderr[-600:]

    pp = os.environ.get("PYTHONPATH","")
    if str(compat) not in pp.split(":"):
        os.environ["PYTHONPATH"] = f"{compat}:{pp}" if pp else str(compat)
    print("PYTHONPATH:", os.environ["PYTHONPATH"])

    # 機能確認: 別プロセスで torchaudio.load が shim（_compat）実装に置換されているか
    chk = subprocess.run([sys.executable,"-c",
        "import torchaudio;torchaudio.set_audio_backend('soundfile');"
        "import inspect;print('shim OK:', inspect.getsourcefile(torchaudio.load))"],
        capture_output=True, text=True, env=os.environ.copy())
    print(chk.stdout.strip() or chk.stderr[-800:])
    assert "shim OK" in chk.stdout and "_compat" in chk.stdout, "★shim が別プロセスに効いていない"


## §4 ローカル運用への展開

fork コードを Drive clone からローカルへ rsync（`Data` / `.git` 除外 = 小ファイル地獄を回避、
`bert/` `slm/` `pretrained_jp_extra/` は少数の大ファイルなので Drive 読みでも数分）、
データは Zenodo からローカルに直接展開、checkpoint / model_assets は Drive へ symlink する。
あわせて fork 固有の `default_style.py` バグ修正と、底モデル（warm-start 元）の配置も行う。


In [ ]:
# ===== §4-A fork コードをローカルへ + default_style.py パッチ =====
import os, shutil, subprocess
from pathlib import Path

ROOT = Path("/content/Style-Bert-VITS2")     # 学習実行ルート（ローカル）
DATA = ROOT / "Data" / CORPUS                # ★コーパスごとに分かれる（§1.2）
CFG  = f"Data/{CORPUS}/config.json"          # SBV2 CLI に渡す相対パス（cwd = ROOT 起点）

# fork コードをローカルへ（初回 数分。Data / .git / model_assets / eval_out は除外）
!rsync -a --info=progress2 --exclude Data --exclude .git --exclude model_assets --exclude eval_out {DRIVE_BASE}/ {ROOT}/
os.chdir(ROOT); print("cwd:", Path.cwd())
assert (ROOT/"train_ms_jp_extra.py").exists(), "★rsync 失敗（DRIVE_BASE を確認）"

# default_style.py の cadseq 巻き込みバグ修正（この fork 固有。upstream には無い）:
# save_neutral_vector / save_styles_by_dirs が rglob("*.npy") で cadence sidecar
# （*.cadseq.npy, 形 (P,32)）まで拾い、256 次元 style と混ざって学習開始直後に
# 「ValueError: dimension 1 ... size 25 vs 40」で落ちる。→ 3箇所に除外ガードを入れる。
def patch_default_style(path):
    p = Path(path)
    src = p.read_text(encoding="utf-8")
    if src.count("cadseq") >= 3:
        print("パッチ適用済み:", p); return
    src = src.replace(
        'for file in wav_dir.rglob("*.npy"):',
        'for file in wav_dir.rglob("*.npy"):\n'
        '        if file.name.endswith(".cadseq.npy"):\n'
        '            continue')
    src = src.replace(
        'npy_files = list(style_dir.rglob("*.npy"))',
        'npy_files = [f for f in style_dir.rglob("*.npy") if not f.name.endswith(".cadseq.npy")]')
    p.write_text(src, encoding="utf-8")
    n = p.read_text(encoding="utf-8").count("cadseq")
    assert n == 3, f"★パッチ結果が想定外（cadseq 出現 {n} ≠ 3）: {p}"
    import py_compile; py_compile.compile(str(p), doraise=True)
    print("パッチ適用:", p, "(cadseq 出現 = 3)")

patch_default_style(ROOT / "default_style.py")         # ローカル（学習が読む方）
patch_default_style(DRIVE_BASE / "default_style.py")   # Drive clone にも当てて永続化（次回 rsync で戻らないように）


In [ ]:
# ===== §4-B データ取得（→ ローカル展開）=====
import json, shutil
from pathlib import Path
# 学習バンドル（tar）を VM ローカル /content に展開する（Drive 直読みの小ファイル I/O 律速を回避）。
# 取得元は §1.2 の CORPORA が決める:
#   fetch="zenodo" → 公開 URL を aria2 16並列で DL（cv_r1）
#   fetch="local"  → DRIVE_BASE/corpus_in/ に置かれた tar を複写（csj_r1。★URL は持たない = 配布しない）
DL = Path("/content/_bundle"); DL.mkdir(exist_ok=True)
CACHE = DRIVE_BASE / "zenodo_cache"     # 公開 DL のキャッシュ（fetch="zenodo" のみ）
# LOCAL_IN（fetch="local" の tar 置き場）は §1.2 で設定済み
CACHE_TO_DRIVE = False                  # True: DL 成功後に Drive へ保存（cv_r1 全量で約 5 GB 消費）

_smoke = C["smoke"]
targets = ([(_smoke[0], _smoke[1], _smoke[2])] if SMOKE else list(C["files"]))

def n_files(d, pat): return sum(1 for _ in Path(d).rglob(pat)) if Path(d).exists() else 0
def _lines(p): return sum(1 for _ in open(p, encoding="utf-8"))

def _placed():
    """★数を焼き込まず、置かれた esd 行数と実 wav 数の一致で判定する（コーパス非依存）。"""
    if not (DATA/"config.json").exists() or not (DATA/"esd_train.list").exists(): return False
    exp = _lines(DATA/"esd_train.list") + _lines(DATA/"esd_val.list")
    if n_files(DATA, "*.wav") != exp: return False
    return (DATA/"SMOKE_MANIFEST.json").exists() == bool(SMOKE)   # 残骸の取り違えを防ぐ

if _placed():
    print("配置済み → 取得 / 展開をスキップ")
else:
    def _ok(p, minsize): return p.exists() and p.stat().st_size >= minsize
    def _fetch(name, url, minsize):
        dst = DL / name
        if _ok(dst, minsize):
            print("取得済み:", name); return
        if dst.exists(): dst.unlink()   # 不完全ファイルは捨てる（低速 DL の再開より並列 DL のほうが速い）
        if C["fetch"] == "local":
            src = LOCAL_IN / name
            assert _ok(src, minsize), (
                f"★入力 tar が無い / 不完全: {src}\n"
                f"   {CORPUS} は自動 DL しない（{C['license']}）。\n"
                f"   dataset_tools/ で作った tar を上記パスに置いてから再実行する。"
                f" 仕様: docs/TRAIN_BUNDLE_SPEC.md")
            print("Drive から複写:", src); shutil.copy2(src, dst); return
        if _ok(CACHE / name, minsize):
            print("Drive キャッシュから複写:", name)
            shutil.copy2(CACHE / name, dst); return
        if not shutil.which("aria2c"):
            !apt-get -qq -y install aria2 > /dev/null
        !aria2c -x16 -s16 -k1M --console-log-level=warn --summary-interval=15 -d {DL} -o {name} "{url}"
        if not _ok(dst, minsize):
            print("★aria2 失敗 → wget にフォールバック")
            !wget -c -O {DL}/{name} "{url}"
        assert _ok(dst, minsize), f"★DL 不完全: {name} (size={dst.stat().st_size if dst.exists() else 0})"
    for name, url, minsize in targets:
        _fetch(name, url, minsize)
    if CACHE_TO_DRIVE and C["fetch"] == "zenodo":
        CACHE.mkdir(exist_ok=True)
        for name, _, minsize in targets:
            if not _ok(CACHE / name, minsize):
                print("Drive へキャッシュ保存:", name); shutil.copy2(DL / name, CACHE / name)
    # --- 展開 → 配置（config.json の位置からデータセットルートを自動判定）---------
    stage = Path("/content/_bundle/x")
    if stage.exists(): shutil.rmtree(stage)
    stage.mkdir(parents=True)
    for name, _, _ in targets:
        print("展開:", name)
        !tar -xf {DL}/{name} -C {stage}
    cfgs = list(stage.rglob("config.json"))
    assert len(cfgs) == 1, f"★config.json の位置を特定できない: {cfgs}"
    src_root = cfgs[0].parent; print("データセットルート検出:", src_root)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    if DATA.exists() and not DATA.is_symlink(): shutil.rmtree(DATA)
    shutil.move(str(src_root), str(DATA))
    if n_files(DATA, "*.wav") == 0:
        # wav tar のルート prefix が meta と異なる場合: 残りを DATA 直下へ統合
        for child in list(stage.iterdir()):
            print("統合:", child.name, "->", DATA/child.name)
            shutil.move(str(child), str(DATA/child.name))

# --- 期待値は「記憶」でなく config.json / esd から読む（§5・§6・§4.5 が参照）--------
_cfg      = json.load(open(DATA/"config.json", encoding="utf-8"))
EXP_TRAIN = _lines(DATA/"esd_train.list")
EXP_VAL   = _lines(DATA/"esd_val.list")
EXP_WAV   = EXP_TRAIN + EXP_VAL          # 1 行 = 1 wav（バンドル仕様）
N_SPK     = len(_cfg["data"].get("spk2id", {}))
N_CADSEQ  = n_files(DATA, "*.cadseq.npy")
print(f"{CORPUS}: 話者 {N_SPK} / esd train {EXP_TRAIN} + val {EXP_VAL} = {EXP_WAV} 行")
print(f"  wav {n_files(DATA, '*.wav')}（{EXP_WAV} が正） / "
      f"cadseq {N_CADSEQ}（カバレッジ {N_CADSEQ/max(EXP_WAV,1):.1%}。コーパスで異なる）")


In [ ]:
# ===== §4-C checkpoint / model_assets の Drive 永続化 + 底モデル配置 =====
import os, shutil
from pathlib import Path

# (1) checkpoint 置き場を Drive へ symlink（読み=ローカル / 保存=Drive。切断後も最新 ckpt から再開できる）
drive_models = DRIVE_BASE/"Data"/CORPUS/"models"; drive_models.mkdir(parents=True, exist_ok=True)
local_models = DATA/"models"
if local_models.exists() and not local_models.is_symlink(): shutil.rmtree(local_models)
!ln -sfn {drive_models} {local_models}
print("models       ->", os.path.realpath(local_models))

# (2) 推論用 model_assets（学習末尾の safetensors 書き出し先）も Drive へ symlink
drive_assets = DRIVE_BASE/"model_assets"; drive_assets.mkdir(exist_ok=True)
local_assets = ROOT/"model_assets"
if local_assets.exists() and not local_assets.is_symlink(): shutil.rmtree(local_assets)
!ln -sfn {drive_assets} {local_assets}
print("model_assets ->", os.path.realpath(local_assets))

# (3) 底モデル（warm-start 元）を models/ へ配置。
#     train は Data/{CORPUS}/models/{G,D,WD}_0.safetensors を warm-start 元として読む。
#     initialize.py は pretrained_jp_extra/ に置くだけなので、コピーしないと
#     ★scratch 学習になり cadence 設計が壊れる（freeze された decoder が初期値のまま凍結される）。
#     既に G_*.pth がある場合は「再開」なのでコピーしない
#     （学習が step 保存時に *_0.safetensors を「古い ckpt」として消すのは正常挙動。原本は pretrained_jp_extra/ に残る）。
resumable = sorted(drive_models.glob("G_*.pth"))
if resumable:
    print("既存 checkpoint あり → 再開モード（底モデルコピー不要）:", resumable[-1].name)
else:
    for f in ("G_0.safetensors","D_0.safetensors","WD_0.safetensors"):
        src = DRIVE_BASE/"pretrained_jp_extra"/f
        assert src.exists(), f"★{src} が無い（setup §3 initialize を先に実行）"
        shutil.copy(src, drive_models/f); print("底モデル配置:", f)


## §4.5 スモーク設定の適用

`SMOKE` の値に応じて config と checkpoint 出力先を切り替える。**SMOKE = False でもこのセルは必ず実行する**
（config の原本復元と検証ゲート用の期待値設定を兼ねる）。esd リストはスモークバンドルに同梱の
サブセット済みのものをそのまま使う（このセルでは加工しない）。config の原本は初回に
`config.full.json` へ退避し、以後は常に原本から再構成するので何度実行しても安全（冪等）。
True ↔ False を切り替えたときは **§4-B から**実行し直す（データ一式が入れ替わる）。


In [ ]:
# ===== §4.5 スモーク設定の適用 =====
import json, re, shutil, subprocess, math
from pathlib import Path

cfgp, cbk = DATA/"config.json", DATA/"config.full.json"
if not cbk.exists(): shutil.copy2(cfgp, cbk)   # 原本退避（初回のみ）。以後は原本から再構成
cyml, cyml_bk = ROOT/"config.yml", ROOT/"config.yml.orig"
if not cyml.exists():
    # config.yml は SBV2 の config.py が初回 import 時に default_config.yml から自動生成する。
    # このセルはどの SBV2 スクリプトより先に走るため、無ければここで同じことをしておく。
    shutil.copy2(ROOT/"default_config.yml", cyml)
if not cyml_bk.exists(): shutil.copy2(cyml, cyml_bk)   # config.yml も原本退避（同上）

MODEL_NAME = f"{CORPUS}_smoke" if SMOKE else CORPUS

if not SMOKE:
    shutil.copy2(cbk, cfgp)
    MDIR = f"Data/{CORPUS}"
    # ★config.yml の model_name が成果物の置き場を決める（out_dir = model_assets/{model_name}、
    #   safetensors 名も {model_name}_e{N}_s{M}.safetensors）。既定のままだと**文字列 "model_name"**
    #   が使われ、model_assets/model_name/ に落ちて全コーパスが混ざる。
    #   （Mac の model_name_e20_s43560.safetensors / Drive の results_model_name_e14_s7000_fixed.json
    #     の「model_name」は、まさにこのバグの痕跡。）原本から書き替えてコーパス名に固定する。
    t = re.sub(r"(?m)^model_name:.*$", f'model_name: "{MODEL_NAME}"',
               cyml_bk.read_text(encoding="utf-8"), count=1)
    assert f'model_name: "{MODEL_NAME}"' in t, "★config.yml に model_name 行が見つからない"
    cyml.write_text(t, encoding="utf-8")
    print(f"SMOKE=False → 全量学習（{CORPUS}。config.json を原本に復元 / model_name を {MODEL_NAME} に固定）")
else:
    man = json.load(open(DATA/"SMOKE_MANIFEST.json", encoding="utf-8"))

    # --- config 上書き（spk2id は原本のまま = 底モデル・embedding 形状に手を触れない）---
    cfg = json.load(open(cbk, encoding="utf-8"))
    cfg["train"]["epochs"]        = SMOKE_EPOCHS
    cfg["train"]["batch_size"]    = SMOKE_BATCH
    cfg["train"]["eval_interval"] = SMOKE_EVAL_INTERVAL
    cfg["train"]["log_interval"]  = 20
    if "model_name" in cfg: cfg["model_name"] = MODEL_NAME   # model_assets を本番と分離
    cap = float(subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip().splitlines()[0])
    if cap < 8.0:  # Ampere 未満 (T4=7.5 等) は bf16 非対応 → fp32 に固定
        cfg["train"]["bf16_run"] = False; cfg["train"]["fp16_run"] = False
        print(f"compute capability {cap} < 8.0 → bf16/fp16 を無効化 (fp32)")
    json.dump(cfg, open(cfgp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    # --- checkpoint 先をスモーク専用ローカルツリーに分離（Drive の本番 models/ を汚さない）---
    MDIR = f"Data/{CORPUS}_smoke"
    # train は -m ディレクトリ直下の wavs/ を直接見る（default_style.save_styles_by_dirs）
    # → データ実体へ symlink（バンドルの wav + style_gen の npy をそのまま参照させる）
    wl = ROOT/MDIR/"wavs"
    wl.parent.mkdir(parents=True, exist_ok=True)
    if not wl.is_symlink():
        if wl.exists(): shutil.rmtree(wl)
        wl.symlink_to(DATA/"wavs")
    # 成果物の分離は config.yml の model_name で決まる（out_dir = model_assets/{model_name},
    # safetensors 命名も同じ。config.json の model_name では変わらない）→ 原本から書き替え
    t = re.sub(r"(?m)^model_name:.*$", f'model_name: "{MODEL_NAME}"',
               cyml_bk.read_text(encoding="utf-8"), count=1)
    assert f'model_name: "{MODEL_NAME}"' in t, "★config.yml に model_name 行が見つからない"
    cyml.write_text(t, encoding="utf-8")
    # 学習済み話者リストを assets 側へ（合成ノートが「有効な話者」を知るため。Drive に永続）
    aout = ROOT/"model_assets"/MODEL_NAME; aout.mkdir(parents=True, exist_ok=True)
    spks = sorted({l.split("|")[1] for l in open(DATA/"esd_train.list", encoding="utf-8")})
    json.dump(spks, open(aout/"trained_speakers.json", "w", encoding="utf-8"), ensure_ascii=False)
    sm = ROOT/MDIR/"models"; sm.mkdir(parents=True, exist_ok=True)
    if not any(sm.glob("G_*.pth")):
        for f in ("G_0.safetensors", "D_0.safetensors", "WD_0.safetensors"):
            shutil.copy2(ROOT/"pretrained_jp_extra"/f, sm/f)
        print("底モデルを", sm, "へ配置")

    steps = math.ceil(EXP_TRAIN/SMOKE_BATCH) * SMOKE_EPOCHS
    assert steps >= SMOKE_EVAL_INTERVAL, (
        f"★総 step ({steps}) が保存間隔 ({SMOKE_EVAL_INTERVAL}) 未満: checkpoint が一度も保存されない。"
        "SMOKE_EVAL_INTERVAL を下げる")
    print(f"スモーク: 話者 {man['n_speakers']} / train {EXP_TRAIN} 行 / val {EXP_VAL} 行 "
          f"/ batch {SMOKE_BATCH} → 総 {steps} step（{SMOKE_EVAL_INTERVAL} step ごと保存）")
print("MDIR =", MDIR, "/ model_assets/", MODEL_NAME)


In [ ]:
# ===== §5 検証ゲート（全 OK になってから先へ進む）=====
# 設計: **ノートに数を焼き込まない。** 期待値は 2 つの出所からしか取らない。
#   (a) バンドル同梱の MANIFEST.json（pack_bundle.py が書いた自己申告）
#   (b) config.json / esd_*.list の実測
# 見るのは「(a) の申告と (b) の実物が一致するか」と「(b) どうしの整合」。
# cv_r1 の性質（298話者・freeze_decoder=True・spk2id 辞書順）を CSJ に持ち込まないこと
# ―― R1 は spk2id が core 0..74 → noncore 75..374 の**非辞書順**、freeze_decoder は
# **False**（凍結は学習コードの requires_grad で決まる）で、どちらも正。
import json, random
import numpy as np
from pathlib import Path

def n_files(d, pat): return sum(1 for _ in Path(d).rglob(pat)) if Path(d).exists() else 0
def _lines(p): return sum(1 for _ in open(p, encoding="utf-8"))

cfg = json.load(open(DATA/"config.json", encoding="utf-8"))
spk2id = cfg["data"].get("spk2id", {})
rows = [l.rstrip("\n").split("|") for l in open(DATA/"esd_train.list", encoding="utf-8")]
esd_spk = {r[1] for r in rows} | {l.split("|")[1] for l in open(DATA/"esd_val.list", encoding="utf-8")}
ckpt_dir = (ROOT/MDIR/"models") if SMOKE else (DATA/"models")
n_wav = n_files(DATA, "*.wav")

# --- (b) 実測どうしの整合（全コーパス共通・これが本体）-----------------------
checks = [
    ("esd: 7 カラム",                                       all(len(r) == 7 for r in rows[:200])),
    (f"config: spk2id {len(spk2id)} 話者 / esd 実出現 {len(esd_spk)}", len(spk2id) > 0),
    ("config: esd の話者が spk2id に全て存在",               esd_spk <= set(spk2id)),
    (f"config: 相対パス (Data/{CORPUS}/…)",                  str(cfg["data"]["training_files"]).startswith(f"Data/{CORPUS}/")),
    (f"wav 実数 {n_wav} ≥ esd 行数 {EXP_WAV}",              n_wav >= EXP_WAV),
    (f"cadseq sidecar {N_CADSEQ} 本（0 < n ≤ {EXP_WAV}）",  0 < N_CADSEQ <= EXP_WAV),
    ("models/ が Drive への symlink",                        (DATA/"models").is_symlink()),
    (f"底モデル or 再開 ckpt あり ({ckpt_dir})",             any(Path(ckpt_dir).glob("G_*"))),
    ("model_assets/ が Drive への symlink",                  (ROOT/"model_assets").is_symlink()),
    ("default_style.py パッチ (cadseq 出現 = 3)",            (ROOT/"default_style.py").read_text(encoding="utf-8").count("cadseq") == 3),
]
# ★config.yml の model_name は全経路で見る（既定のままだと成果物が
#   model_assets/model_name/ に落ちて全コーパスが混ざる。長年この状態だった）
checks.append((f"config.yml の model_name = {MODEL_NAME}（成果物の置き場）",
               f'model_name: "{MODEL_NAME}"' in (ROOT/"config.yml").read_text(encoding="utf-8")))
if SMOKE:
    checks += [
        (f"SMOKE: {MDIR}/wavs が Data/{CORPUS}/wavs への symlink", (ROOT/MDIR/"wavs").is_symlink()),
    ]

# --- (a) MANIFEST の申告 vs 実物（pack_bundle.py 産のバンドルのみ）------------
# cv_r1 の Zenodo バンドルには MANIFEST が無い（2026-07 以前の産物）→ その場合は (b) だけで通す。
man_p = DATA/"MANIFEST.json"
if man_p.exists():
    man = json.load(open(man_p, encoding="utf-8"))
    print(f"MANIFEST: {man['corpus']} / run_id {man['run_id']} / layout {man['layout']}")
    checks += [
        (f"MANIFEST: corpus 名が CORPUS と一致（{man['corpus']}）",  man["corpus"] == CORPUS),
        (f"MANIFEST: esd_train {man['esd_train_lines']} = 実測 {EXP_TRAIN}", man["esd_train_lines"] == EXP_TRAIN),
        (f"MANIFEST: esd_val {man['esd_val_lines']} = 実測 {EXP_VAL}",       man["esd_val_lines"] == EXP_VAL),
        (f"MANIFEST: cadseq {man['cadseq_count']} = 実測 {N_CADSEQ}",        man["cadseq_count"] == N_CADSEQ),
        (f"MANIFEST: 話者 {man['n_speakers_config']} = config 実測 {len(spk2id)}", man["n_speakers_config"] == len(spk2id)),
    ]
    # 申告どまり（assert しない・情報として出す）
    print(f"  申告: spk2id_sorted={man['spk2id_sorted']} / freeze_decoder={man['freeze_decoder']}"
          f" / num_styles={man['num_styles']} / cadseq {man['cadseq_coverage']:.1%}")
    if man["layout"] == "by-speaker":
        print("  ★layout=by-speaker → default_style が話者ごとの style を作る"
              f"（num_styles が {len(spk2id)+1} になる）。cv_r1 / R1 は flat・num_styles=1")
else:
    print("MANIFEST.json 無し（cv_r1 の旧バンドル等）→ 実測どうしの整合のみで検証する")

# 申告でなく実物として確認しておく（assert はしない。コーパスで正しく異なる）
print(f"実測: spk2id 辞書順={list(spk2id) == sorted(spk2id)} / "
      f"freeze_decoder={cfg['train'].get('freeze_decoder')} / num_styles={cfg['data'].get('num_styles')}")
if not cfg["train"].get("freeze_decoder"):
    print("  ※ freeze_decoder が False。R1 はこれが正（凍結は学習コードの requires_grad で決まる）。"
          "cv_r1 は True。学習開始後に「Freezing decoder !!!」が出るかで実際の挙動を確認すること")

ok = True
for name, cond in checks:
    print(("OK " if cond else "★NG"), name); ok &= bool(cond)
assert ok, "★NG を解消してから先へ（§4 / §4.5 を見直す）"

# --- ★cadseq の「存在」でなく「性質」を見る（sampled）------------------------------
# data_utils.py は sidecar を f"{audiopath}.cadseq.npy" で読み、shape が (len(phones), 32)
# でなければ **例外も出さずゼロ系列にフォールバックする**（try/except、警告は最初の1本のみ）。
# = 名前や長さを間違えたバンドルでも学習は「成功」し、cadence が効かないモデルが出来る。
# UTMOS も落ちないので気づけない。新コーパスで最も踏みやすい罠なのでここで潰す。
random.seed(0)
smp = random.sample(rows, min(200, len(rows)))
hit = bad = 0
for r in smp:
    sc = Path(str(ROOT / r[0]) + ".cadseq.npy")     # ★data_utils と同じ解決規則
    if not sc.exists(): continue
    hit += 1
    a = np.load(sc, mmap_mode="r")
    if not (a.ndim == 2 and a.shape[0] == len(r[4].split(" ")) and a.shape[1] == 32):
        bad += 1
        if bad == 1: print("★shape 不一致の例:", sc.name, a.shape, "vs (", len(r[4].split(" ")), ", 32)")
print(f"cadseq sampled: {len(smp)} 行中 {hit} 本ヒット（{hit/len(smp):.0%}）/ shape 不一致 {bad}")
for name, cond in [(f"cadseq: 命名が data_utils と一致（{{wav}}.cadseq.npy が {hit}/{len(smp)}）", hit > 0),
                   ("cadseq: shape = (音素数, 32) が全ヒットで成立",                              bad == 0)]:
    print(("OK " if cond else "★NG"), name); ok &= bool(cond)
assert ok, ("★cadseq が data_utils から見えていない / 形が違う → このまま学習すると "
            "cadence がゼロ系列のまま「成功」する。docs/TRAIN_BUNDLE_SPEC.md を参照")

print(f"\n検証ゲート全通過: {CORPUS}" + ("（スモークモード）" if SMOKE else ""))


In [ ]:
# ===== §6 bert_gen（各 wav の隣に .bert.pt を生成。ローカル I/O + GPU で数分〜10分程度）=====
# 学習バンドルに .bert.pt / style npy は同梱しない → ここで生成する（Drive 直読みだと数時間かかる工程）。
!python bert_gen.py -c {CFG}
import subprocess
n = int(subprocess.run(f"find Data/{CORPUS} -name '*.bert.pt' | wc -l",
                       shell=True, capture_output=True, text=True).stdout.strip() or 0)
print(f"生成された .bert.pt: {n}（esd train {EXP_TRAIN} + val {EXP_VAL} = {EXP_WAV} 以上が正）")
assert n >= EXP_WAV, "★bert.pt 不足 → 上のログを確認（transformers が torch を見失う場合は §2 をやり直す）"


In [ ]:
# ===== §7 style_gen（スタイルベクトル生成。ローカル化で 300 it/s 級 ≈ 数分）=====
# ※ preprocess_text は走らせない（spk2id 再生成の恐れ）。style_gen は esd / config から直接生成する。
#
# ★style_gen.py は **起動時に HF から** pyannote/wespeaker-voxceleb-resnet34-LM を取りに行く
#   （style_gen.py:19、モジュールのトップレベル）。setup の initialize.py は bert / wavlm /
#   pretrained_jp_extra しか Drive に永続化しておらず、このモデルだけ控えが無い。
#   → HF が落ちると学習が止まる（2026-07-16 に LFS resolve 障害で実際に停止）。
#
# ★★置き場は HF_HOME ではない。pyannote の Model.from_pretrained は
#     hf_hub_download(..., cache_dir=CACHE_DIR) と**明示的に渡す**ので HF_HOME は効かない。
#     CACHE_DIR = os.getenv("PYANNOTE_CACHE", "~/.cache/torch/pyannote")  ← これを Drive に向ける。
#   （2026-07-16、HF_HOME だけ設定して「効いていない」ことを実測で確認済み。
#     現物は `python -c "import pyannote.audio.core.model as m; print(m.CACHE_DIR)"` で見られる。）
import os, subprocess
PYA_CACHE = str(DRIVE_BASE / "hf_cache" / "pyannote")
HF_HOME   = str(DRIVE_BASE / "hf_cache")
os.makedirs(PYA_CACHE, exist_ok=True)
_repo = "models--pyannote--wespeaker-voxceleb-resnet34-LM"
_cached = os.path.isdir(f"{PYA_CACHE}/{_repo}")
_off = "HF_HUB_OFFLINE=1" if _cached else ""
print(f"PYANNOTE_CACHE={PYA_CACHE}")
print(f"Drive キャッシュ: {'あり → オフラインで実行（HF が落ちていても通る）' if _cached else 'なし → HF から取得（初回のみ）'}")

# ★`!python` は別プロセスなので環境変数を明示的に渡す（§3 の shim と同じ理由）。
#   HF_HUB_ETAG_TIMEOUT は既定 10 秒で、HF が混むと簡単に超える。
!{_off} PYANNOTE_CACHE={PYA_CACHE} HF_HOME={HF_HOME} HF_HUB_ETAG_TIMEOUT=60 HF_HUB_DOWNLOAD_TIMEOUT=60 python style_gen.py -c {CFG}

# ★style npy は「無くても可」ではない。default_style.save_neutral_vector が
#   np.concatenate(embs) で落ちる（ValueError: need at least one array to concatenate）。
#   style_gen が起動直後に死んでも終了コードを見ないと分からないので、ここで実数を数える。
n = int(subprocess.run(f"find Data/{CORPUS} -name '*.npy' ! -name '*.cadseq.npy' | wc -l",
                       shell=True, capture_output=True, text=True).stdout.strip() or 0)
print(f"style npy: {n}（esd {EXP_WAV} 以上が正）")
assert n >= EXP_WAV, (
    "★style_gen 失敗 → 上のログを確認。HF が原因かは次で切り分ける:\n"
    "   !curl -s -o /dev/null -w '%{http_code}\\n' https://huggingface.co/api/models/pyannote/wespeaker-voxceleb-resnet34-LM\n"
    "   !curl -sL --max-time 20 -o /dev/null -w '%{http_code}\\n' https://huggingface.co/pyannote/wespeaker-voxceleb-resnet34-LM/resolve/main/config.yaml\n"
    "   両方 200 なのに LFS (pytorch_model.bin) だけ 000/504 → HF の LFS 障害。待つ以外にない")

# --- Drive に載ったかを実測する（載っていなければ次回もまた HF に依存する）------------------
print(f"Drive キャッシュ（実行後）:",
      "OK 載った → 次回はオフラインで通る" if os.path.isdir(f"{PYA_CACHE}/{_repo}")
      else "★NG 載っていない → PYANNOTE_CACHE が効いていない。次回も HF 依存")

sv = f"Data/{CORPUS}/style_vectors.npy"
print("style_vectors.npy:", "OK" if os.path.exists(sv)
      else "（未生成でも可: 学習開始時に default_style が上の npy から集約する）")


In [ ]:
# ===== §8 学習 =====
# ・cv_r1 全量: batch=16 / 10 epoch ≈ 8,900〜9,000 step（ローカル運用 ~3.3 it/s ≈ 2時間強、1000 step ごと保存）
#   → step 数は「esd_train 行数 / batch × epochs」で決まる。コーパスを替えれば当然変わる（§4.5 が表示）
# ・スモーク: 既定値で ≈ 300 step（T4 fp32 ~10分、100 step ごと保存。ckpt は Data/{CORPUS}_smoke/models/）
# ・-m は「モデル出力フォルダのパス」。★-m {CORPUS} は誤り（リポジトリ直下の別ツリーに ckpt が落ちる）。
#   正しくは -m Data/{CORPUS}（§4.5 が MDIR に設定済み）
# ・開始直後のログで warm-start を必ず確認する:
#     OK → 「Loaded the pretrained models」+「Missing key: dp/sdp.cadence_cond…, emb_g.weight」群
#          （cadence_cond / emb_g は新規層なので Missing は正常。★emb_g は話者数に依存するので
#            コーパスを替えると形状も変わる = Missing で正しい）
#     NG → 「train from scratch」が出たら即中断 → §4-C（スモーク時は §4.5）の底モデル配置を確認
# ・途中切断したら §1→§5 を再実行してからこのセルを再実行（models/ の最新 checkpoint から自動再開）。
!python train_ms_jp_extra.py -c {CFG} -m {MDIR}


## 出力・再開・運用ノート

| 物 | 場所 | 永続性 |
|---|---|---|
| checkpoint | `Data/{CORPUS}/models/`（→ Drive へ symlink） | Drive に永続 |
| 合成用モデル | `model_assets/{MODEL_NAME}/`（→ Drive へ symlink） | Drive に永続 |
| データ実体 | `/content/Style-Bert-VITS2/Data/{CORPUS}/` | **VM 揮発**（再展開すれば戻る） |
| スモークの ckpt | `Data/{CORPUS}_smoke/models/` | **VM 揮発**（本番 models/ を汚さない） |

- **切断後の再開**: §1 → §1.2 → §5 を再実行してから §8。Drive の最新 checkpoint から自動再開する。
- **コーパスを替えるとき**: §1.2 の `CORPUS` を変えて §1 から通す。`Data/` も `model_assets/` も
  コーパス名で分かれるので、混ざらない（`emb_g` の話者数が違うので、混ざれば即座に落ちる）。
- **★csj_r1 の成果物の扱い**: checkpoint・safetensors・合成音声はいずれも CSJ 由来。
  Drive/HF/Zenodo いずれにも公開しないこと（公開には NINJAL の事前確認が必要）。
  `model_assets/csj_r1/` を共有 Drive に置く場合も共著者限定にする。
- **数字はファイルから読む**: このノートは話者数・行数を一切ハードコードしていない。
  「298 話者」「18,015 発話」などが必要なら `config.json` / `esd_*.list` から読むこと。
